In [1]:
from dataset_generation import *

In [2]:
VERBOSE           = True
SAVE_EVERY        = 100
SIMPLIFY_TIMEOUT  = 10

N_SAMPLES   = 1000
SEED        = random.randint(0, 100000)
OUTPUT_PATH = "datasets/test_taylor_dataset.json"
random.seed(SEED)
cfg = Config(
    n_ops_range  = (3, 6),    # 2–4 internal nodes  ← POC setting
    max_depth    =8,         # max tree depth       ← POC setting
    max_nodes    = 10,         # max total nodes      ← POC setting
    taylor_order = 4,
    x_bias       = 4.0,
)

print("Taylor Series Dataset Generator — POC Edition")
print(f"  Ops      : {BINARY_OPS}  (power: x**2, x**3 only)")
print(f"  Unary    : {UNARY_OPS}")
print(f"  Leaf pool: {LEAF_POOL}  (1 has 3× weight)")
print(f"  Config   : ops={cfg.n_ops_range}  depth≤{cfg.max_depth}  "
        f"nodes≤{cfg.max_nodes}  order={cfg.taylor_order}")
print(f"  Samples  : {N_SAMPLES}   seed={SEED}   save_every={SAVE_EVERY}")
print(f"  Output   : {OUTPUT_PATH}")
print(f"  Timeout  : {SIMPLIFY_TIMEOUT}s per sympy call\n")


Taylor Series Dataset Generator — POC Edition
  Ops      : ['+', '-', '*', '/', '**']  (power: x**2, x**3 only)
  Unary    : ['sin', 'cos', 'exp', 'log', 'sqrt']
  Leaf pool: ['x', '-3', '-2', '-1', '-1', '1', '1', '1', '2', '3']  (1 has 3× weight)
  Config   : ops=(3, 6)  depth≤8  nodes≤10  order=4
  Samples  : 1000   seed=94813   save_every=100
  Output   : datasets/test_taylor_dataset.json
  Timeout  : 10s per sympy call



In [3]:
dataset = generate_dataset(
    cfg         = cfg,
    n_samples   = N_SAMPLES,
    seed        = SEED,
    output_path = OUTPUT_PATH,
)

  [checkpoint] 100/1000 samples  →  datasets/test_taylor_dataset.json
  [checkpoint] 200/1000 samples  →  datasets/test_taylor_dataset.json
  [checkpoint] 300/1000 samples  →  datasets/test_taylor_dataset.json
  [checkpoint] 400/1000 samples  →  datasets/test_taylor_dataset.json
  [checkpoint] 500/1000 samples  →  datasets/test_taylor_dataset.json
  [checkpoint] 600/1000 samples  →  datasets/test_taylor_dataset.json
  [checkpoint] 700/1000 samples  →  datasets/test_taylor_dataset.json
  [checkpoint] 800/1000 samples  →  datasets/test_taylor_dataset.json
  [checkpoint] 900/1000 samples  →  datasets/test_taylor_dataset.json
  [checkpoint] 1000/1000 samples  →  datasets/test_taylor_dataset.json

  Generated : 1000/1000 samples
  Attempts  : 1000  (yield 100.0%)
  Wall time : 28.41s

  Stage averages (over successful attempts):
    generate_fn           2.005 ms  (n=1000)
    taylor_coeffs        22.157 ms  (n=1000)
    prefix_tokens         2.022 ms  (n=1000)


In [4]:
import json
raw_data = json.load(open(OUTPUT_PATH))


In [6]:
for _ in range(50):
    idx = random.randint(0, len(raw_data)-1)
    print(raw_data[idx]["function"]["infix"])

x**2 - x
-3*sin(exp(-x))
x**3
x**3
exp(x**2)
x**3
x**2
-x/2 - 3
exp(x*log(x))
sqrt(x**2 + 2)
x**2
-2/(x**3 - 1)
x**2
x**3
sin(log(sin(x)))
cos(x**3)
x**3 - x
log(sin(x))
-6*x/log(x)
exp(cos(x)) - 2
log(sqrt(x**2))
x**2
-exp(cos(x)) - 1
-x
sin(2/sin(x))
cos(x - 2)
exp(exp(sin(x**3))) - 1
1/(cos(x) + 3)
x**2
cos(x**3)
log(sin(x))
cos(x + 1)
x**3
sqrt(log(exp(x)))
sqrt(x + 5)
log(x) + sin(x) + cos(x) - 1
x**3
sin(4*x)
x**2
log(x**2)
x + 3
sin(exp(x)) - 1
x**2
exp(x**2)/3
log(exp(-3*x))
x*exp(cos(x))
-sin(x) - 7
x - log(x) - 1
(exp(x) + 1)*exp(x)
exp(x)*sin(x)


In [70]:
raw_data[0].keys()

dict_keys(['id', 'function', 'taylor_series', 'metadata'])

In [7]:
complete_seq = []
for func in raw_data:
    coeffs = func["taylor_series"]["coefficients"]
    complete_seq.append(func["function"]["prefix"]+coeffs["coeff0"]["prefix"]+coeffs["coeff1"]["prefix"]+coeffs["coeff2"]["prefix"]+coeffs["coeff3"]["prefix"]+coeffs["coeff4"]["prefix"])



In [10]:
map_count = {0:0, 1:0, 2:0, 3:0, 4:0}
max_tgt=128
count_max_tgt = 0
for seq in complete_seq:
    len_seq = len(seq)
    if len_seq in map_count.keys():
        map_count[len_seq] += 1
    else:
        map_count[len_seq] = 1
    if len_seq<max_tgt:
        count_max_tgt+=1


In [11]:
count_max_tgt

691